# Byte-pair encoding from scratch


In [1]:
# | default_exp tokenisation.bpe

In [2]:
import re
from collections import Counter

import lorem
from tqdm import tqdm

In byte-pair encoding we build a vocabulary by iteratively merging the most frequent pair of adjacent tokens.

1. Start with individual characters as tokens.
2. Find the most frequent pair of adjacent tokens.
3. Merge that pair into a new token.
4. Repeat until we reach the desired size of our vocabulary.


We start off by creating a small corpus of text.


In [3]:
text = lorem.text()

print(text)

Tempora quisquam quisquam porro quiquia dolorem quisquam. Consectetur neque sed non ipsum modi quaerat est. Quisquam eius consectetur tempora consectetur modi sit dolore. Quaerat sit sed quaerat. Numquam ut quiquia quisquam consectetur. Aliquam non aliquam eius dolorem. Magnam aliquam porro labore dolore neque dolor quiquia.

Quiquia est tempora quaerat quaerat dolorem est. Labore dolore labore dolor. Aliquam neque non quaerat numquam numquam dolorem quiquia. Porro quiquia amet modi velit amet. Non adipisci aliquam ut quaerat ipsum etincidunt amet. Est numquam modi adipisci dolore non modi consectetur.

Tempora sed dolor consectetur ipsum porro. Magnam ipsum velit est ipsum modi adipisci. Consectetur porro labore numquam sed amet ipsum. Etincidunt amet quisquam etincidunt etincidunt ipsum. Numquam amet quaerat consectetur magnam. Aliquam porro porro dolor sit ut ut quisquam. Magnam labore amet ut velit quisquam.

Dolor dolor porro porro. Magnam numquam amet dolore tempora consectetur e

In [4]:
tokens = text.encode("utf-8")

print(tokens)

b'Tempora quisquam quisquam porro quiquia dolorem quisquam. Consectetur neque sed non ipsum modi quaerat est. Quisquam eius consectetur tempora consectetur modi sit dolore. Quaerat sit sed quaerat. Numquam ut quiquia quisquam consectetur. Aliquam non aliquam eius dolorem. Magnam aliquam porro labore dolore neque dolor quiquia.\n\nQuiquia est tempora quaerat quaerat dolorem est. Labore dolore labore dolor. Aliquam neque non quaerat numquam numquam dolorem quiquia. Porro quiquia amet modi velit amet. Non adipisci aliquam ut quaerat ipsum etincidunt amet. Est numquam modi adipisci dolore non modi consectetur.\n\nTempora sed dolor consectetur ipsum porro. Magnam ipsum velit est ipsum modi adipisci. Consectetur porro labore numquam sed amet ipsum. Etincidunt amet quisquam etincidunt etincidunt ipsum. Numquam amet quaerat consectetur magnam. Aliquam porro porro dolor sit ut ut quisquam. Magnam labore amet ut velit quisquam.\n\nDolor dolor porro porro. Magnam numquam amet dolore tempora conse

In [5]:
tokens = list(map(int, tokens))

print(tokens)

[84, 101, 109, 112, 111, 114, 97, 32, 113, 117, 105, 115, 113, 117, 97, 109, 32, 113, 117, 105, 115, 113, 117, 97, 109, 32, 112, 111, 114, 114, 111, 32, 113, 117, 105, 113, 117, 105, 97, 32, 100, 111, 108, 111, 114, 101, 109, 32, 113, 117, 105, 115, 113, 117, 97, 109, 46, 32, 67, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 110, 101, 113, 117, 101, 32, 115, 101, 100, 32, 110, 111, 110, 32, 105, 112, 115, 117, 109, 32, 109, 111, 100, 105, 32, 113, 117, 97, 101, 114, 97, 116, 32, 101, 115, 116, 46, 32, 81, 117, 105, 115, 113, 117, 97, 109, 32, 101, 105, 117, 115, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 116, 101, 109, 112, 111, 114, 97, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 109, 111, 100, 105, 32, 115, 105, 116, 32, 100, 111, 108, 111, 114, 101, 46, 32, 81, 117, 97, 101, 114, 97, 116, 32, 115, 105, 116, 32, 115, 101, 100, 32, 113, 117, 97, 101, 114, 97, 116, 46, 32, 78, 117, 109, 113, 117, 97, 109, 32, 117, 116, 32, 113, 117, 105, 113, 1

In [6]:
pair_counts: dict[tuple[int, int], int] = {}
for pair in zip(tokens, tokens[1:]):
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

print(pair_counts)

{(84, 101): 2, (101, 109): 17, (109, 112): 7, (112, 111): 20, (111, 114): 55, (114, 97): 21, (97, 32): 15, (32, 113): 30, (113, 117): 75, (117, 105): 33, (105, 115): 21, (115, 113): 13, (117, 97): 44, (97, 109): 46, (109, 32): 48, (32, 112): 13, (114, 114): 14, (114, 111): 14, (111, 32): 11, (105, 113): 17, (105, 97): 10, (32, 100): 26, (100, 111): 26, (111, 108): 31, (108, 111): 28, (114, 101): 22, (109, 46): 10, (46, 32): 32, (32, 67): 3, (67, 111): 3, (111, 110): 23, (110, 115): 15, (115, 101): 22, (101, 99): 15, (99, 116): 15, (116, 101): 23, (101, 116): 30, (116, 117): 15, (117, 114): 15, (114, 32): 23, (32, 110): 19, (110, 101): 5, (101, 113): 5, (117, 101): 5, (101, 32): 18, (32, 115): 15, (101, 100): 7, (100, 32): 6, (110, 111): 7, (110, 32): 8, (32, 105): 9, (105, 112): 17, (112, 115): 10, (115, 117): 10, (117, 109): 20, (32, 109): 10, (109, 111): 7, (111, 100): 8, (100, 105): 16, (105, 32): 13, (97, 101): 14, (101, 114): 14, (97, 116): 17, (116, 32): 46, (32, 101): 19, (101, 

You can print the sorted pair counts like this:


In [7]:
print(sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True))


[((113, 117), 75), ((111, 114), 55), ((109, 32), 48), ((97, 109), 46), ((116, 32), 46), ((117, 97), 44), ((117, 105), 33), ((46, 32), 32), ((111, 108), 31), ((32, 113), 30), ((101, 116), 30), ((108, 111), 28), ((32, 100), 26), ((100, 111), 26), ((111, 110), 23), ((116, 101), 23), ((114, 32), 23), ((114, 101), 22), ((115, 101), 22), ((114, 97), 21), ((105, 115), 21), ((112, 111), 20), ((117, 109), 20), ((32, 110), 19), ((32, 101), 19), ((32, 97), 19), ((101, 32), 18), ((101, 109), 17), ((105, 113), 17), ((105, 112), 17), ((97, 116), 17), ((100, 105), 16), ((97, 32), 15), ((110, 115), 15), ((101, 99), 15), ((99, 116), 15), ((116, 117), 15), ((117, 114), 15), ((32, 115), 15), ((99, 105), 15), ((114, 114), 14), ((114, 111), 14), ((97, 101), 14), ((101, 114), 14), ((115, 113), 13), ((32, 112), 13), ((105, 32), 13), ((32, 99), 12), ((99, 111), 12), ((111, 32), 11), ((116, 46), 11), ((105, 116), 11), ((105, 97), 10), ((109, 46), 10), ((112, 115), 10), ((115, 117), 10), ((32, 109), 10), ((109,

Next we need to pick the pair with the highest count as our first merge.


In [8]:
most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
print(most_common_pair)
print(f"{chr(most_common_pair[0])}{chr(most_common_pair[1])}")

(113, 117)
qu


And merge it into a new token.


In [9]:
new_token = most_common_pair[0] + most_common_pair[1]
print(new_token)

230


We now update our words to use the new token.


In [10]:
new_tokens = []
i = 0
while i < len(tokens):
    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == most_common_pair:
        new_tokens.append(new_token)
        i += 2
    else:
        new_tokens.append(tokens[i])
        i += 1

print(new_tokens)

[84, 101, 109, 112, 111, 114, 97, 32, 230, 105, 115, 230, 97, 109, 32, 230, 105, 115, 230, 97, 109, 32, 112, 111, 114, 114, 111, 32, 230, 105, 230, 105, 97, 32, 100, 111, 108, 111, 114, 101, 109, 32, 230, 105, 115, 230, 97, 109, 46, 32, 67, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 110, 101, 230, 101, 32, 115, 101, 100, 32, 110, 111, 110, 32, 105, 112, 115, 117, 109, 32, 109, 111, 100, 105, 32, 230, 97, 101, 114, 97, 116, 32, 101, 115, 116, 46, 32, 81, 117, 105, 115, 230, 97, 109, 32, 101, 105, 117, 115, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 116, 101, 109, 112, 111, 114, 97, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 109, 111, 100, 105, 32, 115, 105, 116, 32, 100, 111, 108, 111, 114, 101, 46, 32, 81, 117, 97, 101, 114, 97, 116, 32, 115, 105, 116, 32, 115, 101, 100, 32, 230, 97, 101, 114, 97, 116, 46, 32, 78, 117, 109, 230, 97, 109, 32, 117, 116, 32, 230, 105, 230, 105, 97, 32, 230, 105, 115, 230, 97, 109, 32, 99, 111, 110, 115, 101, 9

---


We can now update our vocabulary with the new token.


In [68]:
vocab.append(new_token)

In [69]:
print(vocab)

['n', 'r', 'a', '.', 'm', 't', 'o', 'e', 'd', 'M', 'p', 'i', 'A', 's', 'u', 'c', 'q', 'l', 'or']


We can also do this clever trick for more succinct code.

This works by joining the words with a separator (e.g. `\x00` because it is not a character that will appear in the text), replacing the pair with the new token, and then splitting the words back apart.

```python
sep = "\x00"
pair_str = sep.join(max_pair)
print(pair_str)
print(sep.join(words[0]))
words = [sep.join(w).replace(pair_str, new_token).split(sep) for w in words]
```


In [ ]:
words

[['M', 'o', 'd', 'i'],
 ['t', 'e', 'm', 'p', 'or', 'a'],
 ['e', 's', 't'],
 ['t', 'e', 'm', 'p', 'or', 'a'],
 ['.'],
 ['A', 'l', 'i', 'q', 'u', 'a', 'm'],
 ['q', 'u', 'a', 'e', 'r', 'a', 't'],
 ['d', 'o', 'l', 'or', 'e'],
 ['n', 'o', 'n'],
 ['a', 'd', 'i', 'p', 'i', 's', 'c', 'i']]

The next step is to repeat this process until we run out of pairs to merge.


In [78]:
vocab = list(set(char for word in words_char_list for char in word))

num_merges = 100
words = [word[:] for word in words_char_list]  # deep copy to preserve original

for _ in tqdm(range(num_merges)):
    pair_counts = Counter(pair for word in words for pair in zip(word[:-1], word[1:]))
    if not pair_counts:
        break
    max_pair = max(pair_counts, key=lambda p: pair_counts[p])
    new_token = max_pair[0] + max_pair[1]
    vocab.append(new_token)

    new_words = []
    for word in words:
        i = 0
        new_word = []
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == max_pair:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_words.append(new_word)
    words = new_words

 33%|███▎      | 33/100 [00:00<00:00, 63725.61it/s]


In [79]:
print(len(vocab))
vocab

51


['n',
 'r',
 'a',
 '.',
 'm',
 't',
 'o',
 'e',
 'd',
 'M',
 'p',
 'i',
 'A',
 's',
 'u',
 'c',
 'q',
 'l',
 'or',
 'di',
 'te',
 'tem',
 'temp',
 'tempor',
 'tempora',
 'qu',
 'qua',
 'Mo',
 'Modi',
 'es',
 'est',
 'Al',
 'Ali',
 'Aliqua',
 'Aliquam',
 'quae',
 'quaer',
 'quaera',
 'quaerat',
 'do',
 'dol',
 'dolor',
 'dolore',
 'no',
 'non',
 'adi',
 'adip',
 'adipi',
 'adipis',
 'adipisc',
 'adipisci']

In [1]:
vocab = {idx: bytes([idx]) for idx in range(256)}

## As a class


In [ ]:
# | export
class BPE:
    def __init__(self, vocab_size: int):
        self.special_tokens = ["<UNK>", "<PAD>", "<BOS>", "<EOS>"]
        self.vocab = []

    def train(self, text: str, vocab_size: int):
        pass

    def encode(self, text: str):
        pass

    def decode(self, tokens: list[int]):
        pass